# MedGemma QLoRA with Unsloth on Kaggle P100 - Low-Memory v14

This notebook fine-tunes `google/medgemma-1.5-4b-it` for 9-class colorectal histology patch classification using NCT-CRC-HE-100K and evaluates on CRC-VAL-HE-7K.

This v14 version fixes the Unsloth `datasets==4.3.0` compatibility issue and imports Unsloth before `datasets` / `transformers` to avoid missing Unsloth patches. It keeps the low-memory Kaggle P100 settings from v13.

Run the dependency cell first. It may restart the kernel once. After restart, run from the top again.



## 1. Dependency bootstrap for Kaggle P100


In [ ]:
# Run this cell first.
# Kaggle's current default PyTorch build may not include sm_60 kernels for Tesla P100.
# This cell repairs the environment, then restarts Python once.

import os
import sys
import subprocess
from pathlib import Path
from importlib import metadata

MARKER = Path('/kaggle/working/.medgemma_unsloth_p100_deps_v14_installed')
CORE_MODULES = ('torch', 'numpy', 'pandas', 'scipy', 'sklearn', 'transformers', 'datasets', 'unsloth')


def pip_install(args):
    print('+', ' '.join(args), flush=True)
    subprocess.check_call(args)


def pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def active_runtime_ok():
    try:
        import torch
        arch = torch.cuda.get_arch_list() if torch.cuda.is_available() else []
        torch_ok = torch.__version__.startswith('2.5.1') and ('sm_60' in arch if torch.cuda.is_available() else True)
        transformers_version = pkg_version('transformers')
        datasets_version = pkg_version('datasets')
        unsloth_version = pkg_version('unsloth')
        transformers_ok = transformers_version is not None and transformers_version.startswith('4.52.')
        datasets_ok = datasets_version == '4.3.0'
        unsloth_ok = unsloth_version is not None
        print('Torch:', torch.__version__)
        print('Torch CUDA runtime:', torch.version.cuda)
        print('Torch CUDA arch list:', arch)
        print('Transformers:', transformers_version)
        print('Datasets:', datasets_version)
        print('Unsloth:', unsloth_version)
        return torch_ok and transformers_ok and datasets_ok and unsloth_ok
    except Exception as exc:
        print('Runtime validation failed:', repr(exc))
        return False


if MARKER.exists() and active_runtime_ok():
    print('Dependency check passed.')
else:
    if MARKER.exists():
        print('Marker exists, but runtime is not compatible. Reinstalling.')
    else:
        print('Installing P100-compatible MedGemma/Unsloth stack.')

    already_loaded = [m for m in CORE_MODULES if m in sys.modules]
    if already_loaded:
        print('Already loaded in this kernel:', ', '.join(already_loaded))
        print('That is OK. Installing now, then restarting so Python reloads packages cleanly.')

    # Stable scientific stack. Keep NumPy < 2 to avoid binary compatibility issues in Kaggle.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.8.4', 'pillow==11.3.0', 'tqdm'
    ])

    # HF / training dependencies. IMPORTANT: Unsloth currently rejects datasets >= 4.4.0.
    # Pin datasets exactly to 4.3.0, then import Unsloth before datasets/transformers later.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'transformers==4.52.3',
        'accelerate>=0.34.0,<1.10.0',
        'bitsandbytes>=0.45.0,<0.47.0',
        'peft>=0.14.0,<0.18.0',
        'trl>=0.18.0,<0.23.0',
        'datasets==4.3.0',
        'huggingface_hub>=0.33.0,<1.0.0',
        'sentencepiece', 'protobuf', 'safetensors'
    ])

    # Install Unsloth without dependencies so it cannot upgrade torch/transformers/datasets behind our back.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
        'unsloth', 'unsloth_zoo'
    ])

    # Re-pin key non-torch packages because other dependencies may have upgraded them.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.8.4', 'pillow==11.3.0', 'datasets==4.3.0'
    ])

    # P100-compatible PyTorch wheel LAST. This is the important part.
    pip_install([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ])

    # Optional xFormers wheel matched to torch 2.5.1. If this fails, continue; Unsloth can fall back.
    try:
        pip_install([
            sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
            'xformers==0.0.28.post3'
        ])
    except Exception as exc:
        print('xformers install failed; continuing without xformers:', repr(exc))

    MARKER.write_text('installed')
    print('\nDependency repair complete. Restarting kernel now.')
    print('After Kaggle restarts, run the notebook from the top again.')
    os._exit(0)




## 2. Imports, constants, metrics helpers


In [ ]:
import os
import re
import json
import time
import math
import shutil
import zipfile
import random
import urllib.request
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:64')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# Import Unsloth before datasets/transformers so its patches are applied.
from unsloth import FastVisionModel, get_chat_template

from huggingface_hub import login
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU compute capability:', torch.cuda.get_device_capability(0))
    print('Torch:', torch.__version__)
    print('Torch arch list:', torch.cuda.get_arch_list())
    if torch.cuda.get_device_capability(0) == (6, 0) and 'sm_60' not in torch.cuda.get_arch_list():
        raise RuntimeError('Active torch does not include sm_60 kernels. Rerun the dependency cell and restart.')

PROJECT_DIR = Path('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v14')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
ADAPTER_DIR = PROJECT_DIR / 'medgemma_crc_unsloth_lora'
for p in [PROJECT_DIR, DATA_DIR, RESULTS_DIR, ADAPTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
LABEL_DESCRIPTIONS = {
    'ADI': 'adipose tissue / fat',
    'BACK': 'empty background / no tissue',
    'DEB': 'debris / necrotic debris',
    'LYM': 'lymphocytes / lymphoid cells',
    'MUC': 'mucus / mucin pools',
    'MUS': 'smooth muscle',
    'NORM': 'normal colon mucosa',
    'STR': 'cancer-associated stroma',
    'TUM': 'colorectal adenocarcinoma epithelium / tumor epithelium',
}
LABEL_TO_DIGIT = {label: str(i) for i, label in enumerate(LABEL_NAMES)}
DIGIT_TO_LABEL = {str(i): label for i, label in enumerate(LABEL_NAMES)}

CLASS_INSTRUCTION = """Classify this H&E colorectal histology image patch.
Return exactly one digit and nothing else.

0 = ADI, adipose tissue / fat
1 = BACK, empty background / no tissue
2 = DEB, debris / necrotic debris
3 = LYM, lymphocytes / lymphoid cells
4 = MUC, mucus / mucin pools
5 = MUS, smooth muscle
6 = NORM, normal colon mucosa
7 = STR, cancer-associated stroma
8 = TUM, colorectal adenocarcinoma epithelium / tumor epithelium

Answer with only one digit from 0 to 8."""


def simple_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred)) if len(y_true) else 0.0


def simple_confusion_matrix(y_true, y_pred, labels):
    label_to_pos = {label: idx for idx, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        if t in label_to_pos and p in label_to_pos:
            cm[label_to_pos[t], label_to_pos[p]] += 1
    return cm


def classification_report_df(y_true, y_pred, labels):
    cm = simple_confusion_matrix(y_true, y_pred, labels)
    rows = []
    total = cm.sum()
    correct = np.trace(cm)
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({'label': label, 'precision': precision, 'recall': recall, 'f1-score': f1, 'support': int(support)})
    macro_f1 = float(np.mean([r['f1-score'] for r in rows])) if rows else 0.0
    weighted_f1 = float(sum(r['f1-score'] * r['support'] for r in rows) / total) if total else 0.0
    acc = float(correct / total) if total else 0.0
    rows.append({'label': 'accuracy', 'precision': acc, 'recall': acc, 'f1-score': acc, 'support': int(total)})
    rows.append({'label': 'macro avg', 'precision': np.mean([r['precision'] for r in rows[:len(labels)]]), 'recall': np.mean([r['recall'] for r in rows[:len(labels)]]), 'f1-score': macro_f1, 'support': int(total)})
    rows.append({'label': 'weighted avg', 'precision': 0.0, 'recall': 0.0, 'f1-score': weighted_f1, 'support': int(total)})
    return pd.DataFrame(rows)


def plot_confusion_matrix(cm, labels, title, output_path):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest')
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        ylabel='True label',
        xlabel='Predicted label',
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    threshold = cm.max() / 2 if cm.size and cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)



## 3. Hugging Face login from Kaggle Secrets


In [ ]:
def get_hf_token():
    token = os.environ.get('HF_TOKEN', '').strip()
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token:
            return token.strip()
    except Exception as exc:
        raise RuntimeError(
            "HF_TOKEN not found. In Kaggle, open Add-ons > Secrets and add a secret named exactly HF_TOKEN. "
            "The token must have access to google/medgemma-1.5-4b-it."
        ) from exc
    raise RuntimeError('HF_TOKEN exists but is empty.')

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
print('Hugging Face login successful.')



## 4. Locate or download NCT-CRC-HE-100K and CRC-VAL-HE-7K


In [ ]:
NCT_CRC_HE_100K_URL = 'https://zenodo.org/records/1214456/files/NCT-CRC-HE-100K.zip'
CRC_VAL_HE_7K_URL = 'https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip'


def looks_like_crc_folder(path: Path) -> bool:
    return path.is_dir() and all((path / label).is_dir() for label in LABEL_NAMES)


def find_crc_folder(folder_name: str):
    roots = [Path('/kaggle/input'), DATA_DIR, Path('/kaggle/working')]
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.glob(f'**/{folder_name}'):
            if looks_like_crc_folder(candidate):
                return candidate
    # Sometimes Kaggle inputs have class folders directly under a dataset folder.
    if folder_name == 'NCT-CRC-HE-100K':
        for candidate in Path('/kaggle/input').glob('*') if Path('/kaggle/input').exists() else []:
            if looks_like_crc_folder(candidate) and 'VAL' not in candidate.name.upper():
                return candidate
    return None


def download_and_extract(url, zip_name, expected_folder):
    existing = find_crc_folder(expected_folder)
    if existing is not None:
        print(f'Found {expected_folder}:', existing)
        return existing

    zip_path = DATA_DIR / zip_name
    if not zip_path.exists():
        print('Downloading:', url)
        print('This may take several minutes for NCT-CRC-HE-100K.')
        urllib.request.urlretrieve(url, zip_path)
        print('Downloaded:', zip_path)
    else:
        print('Using existing zip:', zip_path)

    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)

    extracted = find_crc_folder(expected_folder)
    if extracted is None:
        raise FileNotFoundError(f'Could not locate extracted {expected_folder}. Check zip layout under {DATA_DIR}.')
    print(f'Found {expected_folder}:', extracted)
    return extracted

TRAIN_DIR = download_and_extract(NCT_CRC_HE_100K_URL, 'NCT-CRC-HE-100K.zip', 'NCT-CRC-HE-100K')
TEST_DIR = download_and_extract(CRC_VAL_HE_7K_URL, 'CRC-VAL-HE-7K.zip', 'CRC-VAL-HE-7K')

train_all = load_dataset('imagefolder', data_dir=str(TRAIN_DIR), split='train')
test_dataset = load_dataset('imagefolder', data_dir=str(TEST_DIR), split='train')

train_label_names = train_all.features['label'].names
test_label_names = test_dataset.features['label'].names
print('Train samples:', len(train_all), train_label_names)
print('Test samples:', len(test_dataset), test_label_names)

if train_label_names != LABEL_NAMES:
    print('Warning: dataset label order differs from expected LABEL_NAMES.')
if test_label_names != LABEL_NAMES:
    print('Warning: test label order differs from expected LABEL_NAMES.')



## 5. Training configuration


In [ ]:
# Low-memory Kaggle P100 settings.
# This run is intended to avoid OOM first. Once it works, increase TRAIN_PER_CLASS and MAX_STEPS.

MEDGEMMA_MODEL_ID = 'google/medgemma-1.5-4b-it'

# Start conservative on P100 16GB.
TRAIN_PER_CLASS = 800           # 800 x 9 = 7,200 balanced training examples. Increase to 1500+ only after this works.
MAX_STEPS = 700                 # Increase to 1200-2000 after the smoke test and validation improve.
NUM_TRAIN_EPOCHS = 1            # Used only if MAX_STEPS is None.
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8 # Effective batch 8 without increasing VRAM.
LEARNING_RATE = 1e-4            # Safer than 2e-4 on small/QLoRA vision runs.
LORA_R = 8                      # Lower VRAM than r=16/32.
LORA_ALPHA = 16
SAVE_STEPS = 50                 # Frequent trainer checkpoints so an OOM/restart can resume.
LOGGING_STEPS = 10
MAX_SEQUENCE_LENGTH = 768       # Lower than 2048 to reduce activation memory.

# Evaluation. Start with 1000 for a fast sanity check. Set None for the full 7,180-image benchmark.
EVAL_MAX_SAMPLES = 1000
EVAL_CHECKPOINT_EVERY = 50

print({
    'TRAIN_PER_CLASS': TRAIN_PER_CLASS,
    'MAX_STEPS': MAX_STEPS,
    'NUM_TRAIN_EPOCHS': NUM_TRAIN_EPOCHS,
    'EVAL_MAX_SAMPLES': EVAL_MAX_SAMPLES,
    'LORA_R': LORA_R,
    'GRADIENT_ACCUMULATION_STEPS': GRADIENT_ACCUMULATION_STEPS,
    'MAX_SEQUENCE_LENGTH': MAX_SEQUENCE_LENGTH,
})




## 6. Build a balanced training subset


In [ ]:
def label_name_for_sample(sample, feature_label_names):
    return feature_label_names[int(sample['label'])]


def balanced_indices(dataset, feature_label_names, per_class, seed=SEED):
    rng = random.Random(seed)
    buckets = {label: [] for label in LABEL_NAMES}
    for idx, sample in enumerate(dataset):
        label = label_name_for_sample(sample, feature_label_names)
        if label in buckets:
            buckets[label].append(idx)
    chosen = []
    for label in LABEL_NAMES:
        idxs = buckets[label]
        rng.shuffle(idxs)
        take = len(idxs) if per_class is None else min(per_class, len(idxs))
        chosen.extend(idxs[:take])
        print(f'{label}: selected {take} / {len(idxs)}')
    rng.shuffle(chosen)
    return chosen

train_indices = balanced_indices(train_all, train_label_names, TRAIN_PER_CLASS)
train_subset = train_all.select(train_indices)
print('Training subset size:', len(train_subset))

# Save selected indices for reproducibility.
pd.DataFrame({'train_index': train_indices}).to_csv(RESULTS_DIR / 'selected_train_indices.csv', index=False)



## 7. Conversation dataset with single-token digit targets


In [ ]:
def make_user_message(include_image_placeholder=True):
    content = [{'type': 'text', 'text': CLASS_INSTRUCTION}]
    if include_image_placeholder:
        content.append({'type': 'image'})
    return {'role': 'user', 'content': content}


def convert_sample_to_conversation(sample, feature_label_names):
    label = label_name_for_sample(sample, feature_label_names)
    digit = LABEL_TO_DIGIT[label]
    image = sample['image']
    if hasattr(image, 'convert'):
        image = image.convert('RGB')
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': CLASS_INSTRUCTION},
                    {'type': 'image', 'image': image},
                ],
            },
            {'role': 'assistant', 'content': [{'type': 'text', 'text': digit}]},
        ]
    }


class CRCConversationDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, feature_label_names):
        self.hf_dataset = hf_dataset
        self.feature_label_names = feature_label_names

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        return convert_sample_to_conversation(self.hf_dataset[int(idx)], self.feature_label_names)


train_conversations = CRCConversationDataset(train_subset, train_label_names)
print('Example training record:')
example = train_conversations[0]
print(example['messages'][0]['content'][0]['text'][:300] + '...')
print('target:', example['messages'][1]['content'][0]['text'], DIGIT_TO_LABEL[example['messages'][1]['content'][0]['text']])



## 8. Load MedGemma with Unsloth and add QLoRA adapters


In [ ]:
# This cell is intentionally strict: if Unsloth is incompatible with the active P100 stack,
# stop here instead of silently running a broken training setup.

import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print('Loading with Unsloth FastVisionModel:', MEDGEMMA_MODEL_ID)
model, processor = FastVisionModel.from_pretrained(
    model_name=MEDGEMMA_MODEL_ID,
    load_in_4bit=True,
    token=HF_TOKEN,
    use_gradient_checkpointing='unsloth',
)

# Gemma 3 chat template for vision instruction tuning.
processor = get_chat_template(processor, 'gemma-3')

# Low-memory LoRA: attention adapters only, both vision + language.
# If this still OOMs, set finetune_vision_layers=False and rerun.
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

print('Trainable parameters:')
model.print_trainable_parameters()
if torch.cuda.is_available():
    print('GPU allocated GB after model load:', torch.cuda.memory_allocated() / 1024**3)
    print('GPU reserved GB after model load:', torch.cuda.memory_reserved() / 1024**3)



## 9. Train MedGemma QLoRA with Unsloth


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint

FastVisionModel.for_training(model)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

max_steps_arg = MAX_STEPS if MAX_STEPS is not None else -1
num_train_epochs_arg = NUM_TRAIN_EPOCHS if MAX_STEPS is None else 1
TRAINER_OUTPUT_DIR = PROJECT_DIR / 'trainer_outputs'
TRAINER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_conversations,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        max_steps=max_steps_arg,
        num_train_epochs=num_train_epochs_arg,
        learning_rate=LEARNING_RATE,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_strategy='steps',
        save_total_limit=2,
        optim='paged_adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='cosine',
        seed=SEED,
        output_dir=str(TRAINER_OUTPUT_DIR),
        report_to='none',
        fp16=True,
        bf16=False,
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=MAX_SEQUENCE_LENGTH,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    ),
)

last_checkpoint = get_last_checkpoint(str(TRAINER_OUTPUT_DIR))
if last_checkpoint:
    print('Resuming trainer from checkpoint:', last_checkpoint)
else:
    print('No trainer checkpoint found; starting from step 0.')

start = time.time()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)
train_minutes = (time.time() - start) / 60
print(f'Training finished in {train_minutes:.2f} minutes')
print(train_result)

with open(RESULTS_DIR / 'train_result.json', 'w') as f:
    json.dump({
        'train_minutes': train_minutes,
        'train_result': str(train_result),
        'train_per_class': TRAIN_PER_CLASS,
        'max_steps': MAX_STEPS,
        'learning_rate': LEARNING_RATE,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'max_sequence_length': MAX_SEQUENCE_LENGTH,
    }, f, indent=2)




## 10. Save adapters


In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
print('Saved LoRA adapter and processor to:', ADAPTER_DIR)

adapter_zip = shutil.make_archive(str(PROJECT_DIR / 'medgemma_crc_unsloth_lora_adapter'), 'zip', ADAPTER_DIR)
print('Adapter zip:', adapter_zip)



## 11. Forced-choice digit scoring helpers


In [ ]:
FastVisionModel.for_inference(model)
model.eval()


def move_inputs_to_device(inputs, device=DEVICE):
    moved = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            v = v.to(device)
            # P100 + Gemma 3 vision is more stable with float32 image tensors.
            if k in ('pixel_values', 'images') and torch.is_floating_point(v):
                v = v.to(torch.float32)
        moved[k] = v
    return moved


def build_prompt_inputs(image):
    if hasattr(image, 'convert'):
        image = image.convert('RGB')
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': CLASS_INSTRUCTION},
                {'type': 'image'},
            ],
        }
    ]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors='pt',
    )
    return move_inputs_to_device(inputs)


def get_digit_token_ids(tokenizer):
    mapping = {}
    for digit in [str(i) for i in range(9)]:
        variants = [digit, ' ' + digit, '\n' + digit]
        ids = []
        for v in variants:
            toks = tokenizer(v, add_special_tokens=False).input_ids
            if len(toks) == 1:
                ids.append((v, toks[0]))
        if not ids:
            raise RuntimeError(f'Could not find a single-token encoding for digit {digit}.')
        mapping[digit] = ids
    return mapping

DIGIT_TOKEN_IDS = get_digit_token_ids(processor.tokenizer)
print('Digit token IDs:')
for digit, ids in DIGIT_TOKEN_IDS.items():
    print(digit, ids)


def predict_digit_for_image(image):
    inputs = build_prompt_inputs(image)
    with torch.inference_mode():
        outputs = model(**inputs, use_cache=False, return_dict=True)
        logits = outputs.logits[:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError('Non-finite logits detected during prediction. Stop and lower LR or verify dtype settings.')
        log_probs = torch.log_softmax(logits, dim=-1)[0]

    digit_scores = {}
    for digit, variants in DIGIT_TOKEN_IDS.items():
        # Use the best score among no-space, leading-space, and newline variants.
        scores = [float(log_probs[token_id].item()) for _, token_id in variants]
        digit_scores[digit] = max(scores)

    best_digit = max(digit_scores, key=digit_scores.get)
    pred_label = DIGIT_TO_LABEL[best_digit]
    top = sorted(digit_scores.items(), key=lambda kv: kv[1], reverse=True)
    raw = 'forced_digit_top9=' + ', '.join(f'{DIGIT_TO_LABEL[d]}:{s:.3f}' for d, s in top)
    return pred_label, raw, digit_scores



## 12. Smoke test before full evaluation


In [ ]:
smoke_n = min(24, len(test_dataset))
smoke_rows = []
print(f'Running smoke test on {smoke_n} samples...')
for i in range(smoke_n):
    sample = test_dataset[i]
    true_label = label_name_for_sample(sample, test_label_names)
    pred_label, raw, scores = predict_digit_for_image(sample['image'])
    smoke_rows.append({'sample_index': i, 'true_label': true_label, 'predicted_label': pred_label, 'raw_scores': raw})
    print(f'{i:02d} true={true_label:>5s} pred={pred_label:>5s} {raw[:100]}')

smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(RESULTS_DIR / 'smoke_test_predictions.csv', index=False)
print('Smoke accuracy:', simple_accuracy(smoke_df.true_label, smoke_df.predicted_label))
print('Predicted classes:', sorted(smoke_df.predicted_label.unique().tolist()))

if smoke_df.predicted_label.nunique() <= 1:
    raise RuntimeError('Smoke test collapsed to one class. Do not run full evaluation yet. Try increasing MAX_STEPS or check training loss.')



## 13. Evaluate on CRC-VAL-HE-7K with checkpoints


In [ ]:
eval_dataset = test_dataset
if EVAL_MAX_SAMPLES is not None:
    eval_dataset = eval_dataset.select(range(min(EVAL_MAX_SAMPLES, len(eval_dataset))))
TOTAL_EVAL = len(eval_dataset)
print('Evaluation samples:', TOTAL_EVAL)

PREDICTIONS_PATH = RESULTS_DIR / 'medgemma_unsloth_qlora_predictions.csv'
CHECKPOINT_PATH = RESULTS_DIR / 'medgemma_unsloth_qlora_predictions_checkpoint.csv'

prediction_rows = []
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    prediction_rows = checkpoint_df.to_dict('records')
    processed_indices = set(checkpoint_df['sample_index'].astype(int).tolist()) if len(checkpoint_df) else set()
    print(f'Loaded checkpoint with {len(prediction_rows)} rows. Resuming from {max(processed_indices) + 1 if processed_indices else 0}/{TOTAL_EVAL}.')
else:
    processed_indices = set()
    print('No checkpoint found. Starting evaluation from 0.')

start_time = time.time()
last_checkpoint_time = time.time()

for i in range(TOTAL_EVAL):
    if i in processed_indices:
        continue
    if i % 25 == 0:
        print(f'Processing sample {i}/{TOTAL_EVAL}')

    sample = eval_dataset[i]
    true_label = label_name_for_sample(sample, test_label_names)
    pred_label, raw, digit_scores = predict_digit_for_image(sample['image'])

    row = {
        'sample_index': i,
        'true_label': true_label,
        'predicted_label': pred_label,
        'raw_scores': raw,
    }
    for digit, score in digit_scores.items():
        row[f'score_{DIGIT_TO_LABEL[digit]}'] = score
    prediction_rows.append(row)

    if (i + 1) % EVAL_CHECKPOINT_EVERY == 0:
        pd.DataFrame(prediction_rows).to_csv(CHECKPOINT_PATH, index=False)
        now = time.time()
        checkpoint_minutes = (now - last_checkpoint_time) / 60
        last_checkpoint_time = now
        samples_done = i + 1
        samples_left = TOTAL_EVAL - samples_done
        samples_per_minute = EVAL_CHECKPOINT_EVERY / checkpoint_minutes if checkpoint_minutes > 0 else float('inf')
        eta_hours = samples_left / samples_per_minute / 60 if samples_per_minute > 0 else float('inf')
        print(f'Saved checkpoint: {samples_done}/{TOTAL_EVAL} -> {CHECKPOINT_PATH}')
        print(f'Speed: {samples_per_minute:.2f} samples/min | ETA: {eta_hours:.2f} hours')

    if torch.cuda.is_available() and (i + 1) % 25 == 0:
        torch.cuda.empty_cache()

results_df = pd.DataFrame(prediction_rows).sort_values('sample_index')
results_df.to_csv(PREDICTIONS_PATH, index=False)
results_df.to_csv(CHECKPOINT_PATH, index=False)
print('Saved predictions:', PREDICTIONS_PATH)
print('Evaluation minutes:', (time.time() - start_time) / 60)
results_df.head()



## 14. Metrics, reports, confusion matrix, and zip


In [ ]:
results_df = pd.read_csv(RESULTS_DIR / 'medgemma_unsloth_qlora_predictions.csv')
y_true = results_df['true_label'].tolist()
y_pred = results_df['predicted_label'].tolist()

accuracy = simple_accuracy(y_true, y_pred)
report = classification_report_df(y_true, y_pred, LABEL_NAMES)
cm = simple_confusion_matrix(y_true, y_pred, LABEL_NAMES)

metrics = {
    'model': 'medgemma_unsloth_qlora_digit_target_lowmem_v14',
    'base_model': MEDGEMMA_MODEL_ID,
    'kaggle_gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'num_eval_samples': int(len(results_df)),
    'accuracy': accuracy,
    'macro_f1': float(report.loc[report['label'] == 'macro avg', 'f1-score'].iloc[0]),
    'weighted_f1': float(report.loc[report['label'] == 'weighted avg', 'f1-score'].iloc[0]),
    'train_per_class': TRAIN_PER_CLASS,
    'max_steps': MAX_STEPS,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'learning_rate': LEARNING_RATE,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'eval_max_samples': EVAL_MAX_SAMPLES,
}

with open(RESULTS_DIR / 'medgemma_unsloth_qlora_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
report.to_csv(RESULTS_DIR / 'medgemma_unsloth_qlora_classification_report.csv', index=False)
pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES).to_csv(RESULTS_DIR / 'medgemma_unsloth_qlora_confusion_matrix.csv')

plot_confusion_matrix(cm, LABEL_NAMES, 'MedGemma Unsloth QLoRA Confusion Matrix', RESULTS_DIR / 'medgemma_unsloth_qlora_confusion_matrix.png')

print(json.dumps(metrics, indent=2))
print(report)

# Save a comparison template. Add your ViT numbers manually if needed.
comparison = pd.DataFrame([
    {'Model': 'Previous MedGemma QLoRA', 'Accuracy': 0.26016713091922006, 'Notes': 'Uploaded previous run'},
    {'Model': 'MedGemma Unsloth QLoRA digit target lowmem v14', 'Accuracy': accuracy, 'Notes': f'NCT balanced train_per_class={TRAIN_PER_CLASS}, max_steps={MAX_STEPS}, eval_max={EVAL_MAX_SAMPLES}'},
])
comparison.to_csv(RESULTS_DIR / 'comparison_summary.csv', index=False)

zip_path = shutil.make_archive(str(PROJECT_DIR / 'medgemma_unsloth_qlora_results'), 'zip', RESULTS_DIR)
print('Created results zip:', zip_path)

# Kaggle FileLink sometimes 404s for absolute paths. This gives a relative link from /kaggle/working.
os.chdir('/kaggle/working')
try:
    from IPython.display import FileLink, display
    display(FileLink('MedGemma_Unsloth_QLoRA_CRC_lowmem_v14/medgemma_unsloth_qlora_results.zip'))
except Exception as exc:
    print('FileLink unavailable:', exc)
    print('Download manually from:', zip_path)




## 15. Optional: save a committed Kaggle output


In [ ]:
print('Important: before ending the session, download this zip or click Save Version / Commit in Kaggle:')
print('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v14/medgemma_unsloth_qlora_results.zip')
print('Adapter zip:')
print('/kaggle/working/MedGemma_Unsloth_QLoRA_CRC_lowmem_v14/medgemma_crc_unsloth_lora_adapter.zip')

